In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 1. Parameter Utama
tahun = 2026
BASE_DIR = os.getcwd()

url_sumber = f"https://data.chc.ucsb.edu/products/CHIRPS/v3.0/daily/final/rnl/{tahun}/"
folder_simpan = os.path.join(BASE_DIR, "data", "CHIRPS", str(tahun))

# Buat folder jika belum ada
os.makedirs(folder_simpan, exist_ok=True)

print(f"Folder output: {folder_simpan}")

# 2. Ambil daftar file dari website
print(f"Membaca isi direktori: {url_sumber}")
response = requests.get(url_sumber, timeout=60)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# 3. Cari semua link (tag <a>)
for link in soup.find_all("a"):
    nama_file = link.get("href")

    if not nama_file or nama_file in ("../", "/"):
        continue

    # Filter: Hanya unduh file dengan ekstensi tertentu (misal .tif atau .tif.gz)
    if nama_file.endswith(".tif") or nama_file.endswith(".tif.gz"):
        link_download = urljoin(url_sumber, nama_file)
        path_simpan = os.path.join(folder_simpan, os.path.basename(nama_file))

        # Skip jika file sudah pernah diunduh (agar bisa di-pause/resume)
        if not os.path.exists(path_simpan):
            print(f"Mendownload: {nama_file} ...", end="")
            with requests.get(link_download, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(path_simpan, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
            print(" Selesai!")
        else:
            print(f"File {nama_file} sudah ada, lanjut ke file berikutnya.")

print("\n🎉 Semua file berhasil diunduh!")